# stock position and pricing reconcilation model

Automated model that ensures consistency and accuracy across different systems, records or counterparties regarding:

* *Reconciles what the portfolio holds vs what the porfolio is worth*

# This project's purpose:
---
objectives: 

* Three-way book of records vs custodian vs independent market data reconciliation model.

* Multi-asset class portfolio - comprising equities, fixed income assets, funds on Net Asset Value.

* A corporate actions adjustment layer - to account for stock splits / dividends / shareholder rights issues leading to the model breaking.

* Dynamic tolerance thresholds 

* Automated break-cause classification

* Service level Agreement (SLA) aging with Current Account Switch Service (CASS) style excalation tiers.

* *model should be fully auditable*


# Milestones:

- M1: Data model design - schema for entities
- M2: build the data generator, load into a database
- M3: Core thre-way matching logic + corporate action adjustment
- M4: Break classification, aging/ escalation, audit log
- M5: Power BI/ Excel dashboard on top of the engine outpot
- M6: Write up: README, architecture diagrame, screenshots.

# Architecture

* Data model (portfolio structure)
* Engine (Python + SQL)
* Dashboard (PowerBI / Excel)


## layer 1 - Data model: Book of Records, Custodian Statement, Market Data Vendor, Corporate Actions.

### 1. Book of Records: based on Investment book of records

Refered to as Investment Book of Records (IBOR)
* security identification (ISIN and security name columns)

corresponds directly to the holdings section of the custodian statement. with every position tagged with its ISIN so that when daily custodian feed arives, each line can be matched automatically. 

* Asset class column - (UK Equity, US Equity, Fixed Income, ETF, Cash)

* currency column - (GBP, USD,EUR)

* Quantity column - units

* Average cost price and cost basis columns

* market price and price source columns

* Market value columns

* unrealised P&L column

* weight(%) position

reconciliation layer: 
* custodian quantity - pulled from the custodian's daily holdings feed

* quantity variance - IBOR quantity minus the custodian quantity

* custodian Market value - the custodian's quantity priced at the same market price (to isolate position differences from pricing differences)

* value variance columns - captures any residual leakage.


bottom rows:
Cash row - maps to the custodian's cash balance section.
total rows - summing all market values and gives the total portfolio value, which is the denominator for the weight calculation and the basis for NAV (Net asset value)

### 2. Custodian statement: formal report issued by a custodian bank (eg. BNY Mellon, State Street, J.P.Morgan, North Trust) to an asset management firm, detailing everything the custodian holds or has processed on that firm's behalf ocer a given period.

recall the custodian is the instituion that physically safeguards the fund's assets (securities and cash), settles trades, collects income, and processes corporate actions. categories of information include: 

* the holdings or positions section lists with every security held, identified by ISIN or SEDOL, with quantity, the market in which it's held, and often a valuation using the custodian's pricing source.

* the cash balance section shows money held across each currency account, including settle and unsettled amounts.

* the transaction section records all movements during the period - purchases, sales, settlements, transfers in and out.

* the income section details dividends received, coupon payments collected, and any tax withheld at source.

* the corporate actions section shows events processed, such as stock splits, bonus issues, rights taken up, and how entitlements were credited. 

* statements delivered daily, monthly, or at other intervals depending on the agreement between the firm and the custodian - usually daily.


### 3. Market data vendor

***dataset needs to be either modelled similarly to a real data vendor or designed from scratch as API keys cost up to £30,000*

As the price attached to a security determines the value of the entire portfolio. if the price is wrong, the NAV (Net Asset Value) is wrong, client reports are wrong, and the fund may be trading on a false picture of its own worth. 

* security identification (ISIN security name, and CCY columns)
currency matters as the price quoted in USD or EUR must be converted at an appropriate Forex rate (FX rate) to reach the GBP (given a uk based investment company) market value 

* Bloomberg close price column - given its the primary vendor
the number that flows into the IBOR's market price column - the investment management firm's book of records price for the asset held - bloomberg is the most widely used market data terminal (hence all the bloomberg terminals at investment firms)

*by making it the primary source of data mmeans the firm has made a documented decision under FUND 3.9 THAT BLOOMBERGS's closing prices are fair, reliable and independednt for the asset classess the fund holds.

*consideration* - replicate using yfinance api for close price on different asset classes.

* Refinitiv close price column (secondary vendor)

independent cross-check - avoids concetration risk from single pricing source.

----
*CASS 6.6 requires the asset manager to reconcile its own internal book of records against the custodians's records on a regular basis. and daily if assets are held electronically at a central securities depository. when the two don't agree, that's a "break" and the firm must investigate, document, and resolve it.*

Required to follow Financial Conduct Authority (FCA) mandates primarily:
FCA: https://handbook.fca.org.uk/handbook/sysc9/sysc9s1 



## relevant regulations 



 1. SYSC 9 - General rules of Record-Keeping. 


SYSC 9.1.1. Firms must arrage for orderly records to be kept of its business and internal organisation, including all services and transactions undertaken by it, which must be sufficient to enable FCA to monitor the firm's compliance with the requirements under the regularatory system, and in particular to ascertain that the firm has complied with all obligations with respect to clients. 

- 9.1.1B - risk management records, the firm's business and internal organisation, compliance reports, conflict of interest record, internal audit reports, records of personal transactions.

SYSC 9.1.2. A common platform firm must retain all records kept by it under this chapter in relation to its MiFID business for a period of at least *5 years* 

- 9.1.2A - retain records in a medium that allows the storage of information in a way accessible for future reference by the FCA.

 *the FCA or any other relevant competent authority must be able to access them readily and to reconstiture each key stage of the processing of each transation*

 *it must be possible for any corrections or other ammendments, and the contents of the records prior to such corrections and ammendments, to be easily ascertained.*

 *it must not be possible for the records otherwise to be manipulated or altered*

 *it must allow the use of IT systems or any other efficient technology when the analysis of the data cannot be easily carried out due to the volume and the nature of the data*
 
 *the firm's arrangements must comply with the record keeping requirements irrespective of the technology used*

2. SYSC 10A - Recording Telephone conversations and electronic communications - (*not relevant for project*)

3. COBS 9.5/ COBS 9A.4 - suitablility records - (*directly not applicable but useful to know*)

When a firm provides investment advice to a retail client in the course of MiFID business, it must keep a record of the time and date the advice was given, the financial instrument recommended, and the suitabilility report provided to the client. For life policies, personal pension schemes, stakeholder pension schemes, or defined contribution occupational pension scheme benefits, suitablility records must be retained for five years. Pension transfer suitablility records must be kept idenfinetly under COBS 9.5.2R

4. COBS 11.5A - Record keeping for client orders and transactions

Firms must, immediately after receiving a client order or making a decision to deal, record and keep at the disposal of the FCA at least the prescibed details, including a designation to identify the trader responsible for the investment decision. Orders executed on behalf of clients must be promptly and accurately recorded and allocated. 

5. CASS 6 - Client Assets (safe Custody) Records
https://handbook.fca.org.uk/handbook/cass6 

To comply with CASS 6, FCA - regulated firms must maintain client-specific custody asset records, undertake an internal custody check at least once a month, and carry out an external custody reconciliation at least monthly or daily where client assets are held electronically with a central securities depository. Firms must introduce adequate organisational arrangements to minimise the risk of loss or diminution of clients' safe custody assets as a result of misuse, fraud, poor administration, inadequate record-keeping or negligence. 

6. CASS 7 - Client Money Records

Firms must perform an internal client money reconcilation each business day, reconciling the firm's own records of client money balances against client transaction records. Requirements cover allocation of the supervisory function over CASS compliance, client money calculations, reconcilations, client communications, and details of mandates held

7. DISP 1.9 - complaints Records (*not applicable for this project*)

8. Money Laundering Regulations 2017 (regulation 40) - AML/KYC records

## summary

summary:

|Record Type	|Minimum Retention	|Source|
|---|---|---|
|General MiFID business records|	5 years	|SYSC 9.1.2R|
|Suitability assessments (general)|	5 years	|COBS 9.5.2R|
|Pension transfer suitability	|Indefinite	|COBS 9.5.2R|
|Appropriateness assessments	|5 years	|COBS 10.7|
|Client order/transaction records	|5 years	|COBS 11.5A|
|Telephone/electronic comms|	5 years (up to 7 on FCA request)|	SYSC 10A|
|Financial promotions	|5 years	|COBS 4.11 / Sch 1|
|Complaints	|3 years	|DISP 1.9.1R|
|AML/CDD documents	|5 years post-relationship end	|MLR 2017 Reg. 40|
|Client asset records (CASS)	|Duration of custody + reconciliation logs|	CASS 6|
|Consumer Duty outcome evidence|	Duration of relevance	|PRIN 2A|
| market vendor| documentation | FUND 3.9|